# MedSignal – Phase I
## openFDA MAUDE API: Daten ziehen, verstehen, vorbereiten

**Ziel dieses Notebooks:**
- openFDA API kennenlernen
- MAUDE Adverse Events für einen Gerätetyp ziehen
- EDA: Struktur, Felder, Klassenverteilung
- Datensatz für ML speichern

**Doku:** https://open.fda.gov/apis/device/event/

In [ ]:
import requests
import pandas as pd
import json
import time
import seaborn as sn
import matplotlib.pyplot as plt
from pathlib import Path

# Ordner anlegen
Path('data').mkdir(exist_ok=True)
Path('data/raw').mkdir(exist_ok=True)

BASE_URL = 'https://api.fda.gov/device/event.json'
print('Setup OK')

## 1. API Grundstruktur verstehen

Die openFDA API hat 3 Hauptparameter:
- `search`: Filterabfrage (Lucene-Syntax)
- `count`: Aggregation nach Feld
- `limit` / `skip`: Pagination (max. 1000 pro Request)

In [ ]:
# --- Schritt 1: Wie viele Einträge gibt es für Insulinpumpen? ---
r = requests.get(BASE_URL, params={
    'search': 'device.generic_name:"insulin pump"',
    'limit': 1
})
total = r.json()['meta']['results']['total']
print(f'Gesamtzahl Insulinpumpen-Events: {total:,}')

In [ ]:
# --- Schritt 2: Verteilung der Event-Typen ---
r = requests.get(BASE_URL, params={
    'search': 'device.generic_name:"insulin pump"',
    'count': 'event_type.exact',
})
event_types = pd.DataFrame(r.json()['results'])
print('Event Types:')
print(event_types)

In [ ]:
# --- Schritt 3: Häufigste Problem-Codes ---
r = requests.get(BASE_URL, params={
    'search': 'device.generic_name:"insulin pump"',
    'count': 'device.device_report_product_code.exact',
    'limit': 10
})
print('Top 10 Product Codes:')
print(pd.DataFrame(r.json()['results']))

In [ ]:
# --- Schritt 4: Zeitlicher Trend der Meldungen ---
r = requests.get(BASE_URL, params={
    'search': 'device.generic_name:"insulin pump"',
    'count': 'date_of_event',
})


In [ ]:
df_trend = pd.DataFrame(r.json()['results'])
df_trend['date'] = pd.to_datetime(df_trend['time'], format='%Y%m%d')
df_trend = df_trend.set_index('date').sort_index()

df_yearly = df_trend['count'].resample('YE').sum().reset_index()
df_yearly.columns = ['date', 'count']
#df_trend.isna().sum()
df_yearly.head()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
sn.lineplot(data=df_yearly, x='date', y='count', ax=ax)
ax.set_title('Insulinpumpen Adverse Events pro Jahr')
ax.set_ylabel('Anzahl Meldungen')
ax.set_xlabel('')
plt.tight_layout()
plt.savefig('data/trend_insulinpumpen.png', dpi=150)
plt.show()

## 2. Daten im Bulk ziehen

openFDA erlaubt max. **1000 Records pro Request** und **1000 Requests/Tag** ohne API Key.
Mit API Key (kostenlos, registrieren auf open.fda.gov) sind 120.000 Requests/Tag möglich.

Strategie: Pagination mit `skip`-Parameter.

In [ ]:
# --- API Key (optional aber empfohlen) ---
# Registrierung: https://open.fda.gov/apis/authentication/
!pip install dotenv
from dotenv import load_dotenv
import os

load_dotenv()
API_KEY = os.getenv('OPENFDA_API_KEY', '')
print('Key geladen:', bool(API_KEY))


In [ ]:
def fetch_maude_by_daterange(search_base, start_year=2015, end_year=2025, api_key=''):
    all_records = []

    for year in range(start_year, end_year + 1):
        # Leerzeichen statt + um TO
        search = f'{search_base} AND date_of_event:[{year}0101 TO {year}1231]'
        skip = 0

        while True:
            params = {
                'search': search,
                'limit': 1000,
                'skip': skip
            }
            if api_key:
                params['api_key'] = api_key

            r = requests.get(BASE_URL, params=params, timeout=30)

            if r.status_code != 200:
                print(f'\n{year} Fehler: {r.status_code} – {r.json().get("error", {}).get("message", "")}')
                break

            batch = r.json().get('results', [])
            if not batch:
                break

            all_records.extend(batch)
            skip += len(batch)
            print(f'{year} | {skip:>5} Records...', end='\r')
            time.sleep(0.1)

        print(f'{year} | {skip:>5} Records geladen')

    print(f'\nGesamt: {len(all_records):,} Records')
    return all_records

In [ ]:
# Einzeltest Jahr 2023
r = requests.get(BASE_URL, params={
    'search': 'device.generic_name:"insulin pump" AND date_of_event:[20230101 TO 20231231]',
    'limit': 1,
    'api_key': API_KEY
})
print(r.status_code)
print(r.json()['meta']['results']['total'])

In [ ]:
#ok... jetzt alles:
raw_data = fetch_maude_by_daterange(
    search_base='device.generic_name:"insulin pump"',
    start_year=2015,
    end_year=2025,
    api_key=API_KEY
)

## 3. JSON → DataFrame flachklopfen

MAUDE-Records sind verschachtelt. Diese Felder sind für ML relevant:

In [ ]:
def flatten_maude_record(rec):
    device  = rec.get('device',  [{}])
    device  = device[0]  if device  else {}
    patient = rec.get('patient', [{}])
    patient = patient[0] if patient else {}
    mdr     = rec.get('mdr_text', [{}])
    mdr     = mdr[0]     if mdr     else {}

    outcome_list = patient.get('sequence_number_outcome', [])

    return {
        'report_number':     rec.get('report_number'),
        'date_received':     rec.get('date_received'),
        'date_of_event':     rec.get('date_of_event'),
        'generic_name':      device.get('generic_name'),
        'brand_name':        device.get('brand_name'),
        'manufacturer_name': device.get('manufacturer_d_name'),
        'device_age_text':   device.get('device_age_text'),
        'product_code':      device.get('device_report_product_code'),
        'implant_flag':      device.get('implant_flag'),
        'single_use_flag':   device.get('single_use_flag'),
        'event_type':        rec.get('event_type'),
        'adverse_event_flag':rec.get('adverse_event_flag'),
        'report_source':     rec.get('report_source_code'),
        'reporter_country':  rec.get('reporter_country_code'),
        'patient_outcome':   outcome_list[0] if outcome_list else None,
        'narrative_text':    mdr.get('text', ''),
    }



In [ ]:
df = pd.DataFrame([flatten_maude_record(r) for r in raw_data])
df.to_parquet('data/raw/maude_pass1.parquet', index=False)
import os
os.chdir('..')  #vigilex/notebooks -> vigilex
Path('data/raw').mkdir(parents=True, exist_ok=True)
df.to_parquet('data/raw/maude_pass1.parquet', index=False)
print('Gespeichert:', Path('data/raw/maude_pass1.parquet').stat().st_size / 1e6, 'MB')

In [ ]:
print(f'Shape: {df.shape}')
df.head(3)

In [ ]:
# --- Schnell-EDA ---
print('=== Dtypes ===')
print(df.dtypes)

print('\n=== Missing Values (%) ===')
print((df.isnull().mean() * 100).round(1).sort_values(ascending=False))

print('\n=== event_type Verteilung (ML Target 1) ===')
print(df['event_type'].value_counts())

print('\n=== patient_outcome Verteilung (ML Target 2) ===')
print(df['patient_outcome'].value_counts())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Event Types
df['event_type'].value_counts().plot(
    kind='bar', ax=axes[0], color='steelblue', title='Event Type Verteilung'
)
axes[0].set_xlabel('')

# Patient Outcomes
df['patient_outcome'].value_counts().head(8).plot(
    kind='bar', ax=axes[1], color='coral', title='Patient Outcome Verteilung'
)
axes[1].set_xlabel('')

plt.tight_layout()
plt.savefig('data/eda_verteilungen.png', dpi=150)
plt.show()

In [ ]:
# --- Top 10 Hersteller ---
df['manufacturer_name'].value_counts().head(10).plot(
    kind='barh', figsize=(10, 5),
    title='Top 10 Hersteller nach Meldungsanzahl',
    color='steelblue'
)
plt.tight_layout()
plt.show()

print('\nTop Hersteller:')
print(df['manufacturer_name'].value_counts().head(10))

## 4. Daten speichern

Rohdaten als Parquet (effizient) und CSV (lesbar) speichern.

In [ ]:
# Datumsspalten parsen
for col in ['date_received', 'date_of_event']:
    df[col] = pd.to_datetime(df[col], format='%Y%m%d', errors='coerce')

# Speichern
df.to_parquet('data/raw/maude_insulinpumpen.parquet', index=False)
df.to_csv('data/raw/maude_insulinpumpen.csv', index=False)

# Raw JSON als Backup
with open('data/raw/maude_raw.json', 'w') as f:
    json.dump(raw_data, f)

print(f'Gespeichert: {len(df):,} Records')
print(f'Parquet: data/raw/maude_insulinpumpen.parquet')
print(f'CSV:     data/raw/maude_insulinpumpen.csv')

## 5. Recall Database kreuzen (ML Target: Recall Ja/Nein)

FDA Recall Database ebenfalls über openFDA verfügbar.
Diese wird genutzt um zu labeln: Hat ein Gerät nach X Complaints einen Recall bekommen?

In [ ]:
RECALL_URL = 'https://api.fda.gov/device/recall.json'

# Recalls für Insulinpumpen abrufen
r = requests.get(RECALL_URL, params={
    'search': 'product_description:"insulin pump"',
    'limit': 100
})

if r.status_code == 200:
    recalls = pd.DataFrame(r.json()['results'])
    print(f'Recalls gefunden: {len(recalls)}')
    print(recalls[['recall_number', 'product_description',
                   'recalling_firm', 'recall_initiation_date',
                   'classification']].head(5))
else:
    print(f'Fehler: {r.status_code}')

In [ ]:
# Recall-Klassen erklären:
# Class I  = höchstes Risiko (Lebensgefahr)
# Class II = mittleres Risiko (temporäre Beeinträchtigung)
# Class III= geringstes Risiko (kein Gesundheitsrisiko)

if 'recalls' in dir() and len(recalls) > 0:
    print('Recall Klassen (dein Domain-Wissen: wie bei SAE-Klassifikation):')
    print(recalls['classification'].value_counts())

    # Hersteller mit Recalls
    print('\nHersteller mit Recalls:')
    print(recalls['recalling_firm'].value_counts().head(10))

## Nächste Schritte (Notebook 02)

1. Features bauen: Rolling-Window Complaint-Counts, Schweregrad-Score
2. Target labeln: Recall innerhalb 12 Monate nach erstem Complaint-Peak
3. Baseline Modelle: LogReg, RandomForest
4. Evaluation: Recall-fokussiert (kein FN-Risiko bei Patientensicherheit)

**Wichtig:** Dein DKFZ-Background hilft hier direkt bei der Feature-Interpretation –
du weißt welche Complaint-Patterns in der Praxis auf ein echtes Problem hindeuten.